#### Enigma Part

* basically we need it to encrypt our cribs
* which means that it encrypts the cribs and then checks back with the bombe
* keeps doing this so that it can find whats right and whats wrong
* without this part its like having a key but no lock to put it in

In [1]:
import time
import string
from typing import List, Tuple, Optional, Dict, Union
from itertools import product
from datetime import datetime
import os


class EnigmaMachine:
    """3-rotor Enigma machine implementation"""
    
    ROTORS = {
        'I':   'EKMFLGDQVZNTOWYHXUSPAIBRCJ',
        'II':  'AJDKSIRUXBLHWTMCQGZNPYFVOE',
        'III': 'BDFHJLCPRTXVZNYEIWGAKMUSQO',
        'IV':  'ESOVPZJAYQUIRHXLNFTGKDCMWB',
        'V':   'VZBRGITYUPSDNHLXAWMJQOFECK'
    }
    
    NOTCHES = {'I': 'Q', 'II': 'E', 'III': 'V', 'IV': 'J', 'V': 'Z'}
    REFLECTOR_B = 'YRUHQSLDPXNGOKMIEBFZCWVJAT'
    ALPHABET = string.ascii_uppercase
    
    def __init__(self, rotors: Tuple[str, str, str], positions: Tuple[int, int, int],
                 ring_settings: Tuple[int, int, int] = (0, 0, 0),
                 plugboard: Dict[str, str] = None):
        self.rotors = [self.ROTORS[r] for r in rotors]
        self.rotor_names = rotors
        self.positions = list(positions)
        self.ring_settings = list(ring_settings)
        self.notches = [self.NOTCHES[r] for r in rotors]
        self.reflector = self.REFLECTOR_B
        self.plugboard = plugboard or {}
    
    def _plugboard_swap(self, char: str) -> str:
        return self.plugboard.get(char, char)
    
    def _rotate_rotors(self):
        if self.ALPHABET[self.positions[1]] == self.notches[1]:
            self.positions[1] = (self.positions[1] + 1) % 26
            self.positions[2] = (self.positions[2] + 1) % 26
        elif self.ALPHABET[self.positions[0]] == self.notches[0]:
            self.positions[1] = (self.positions[1] + 1) % 26
        self.positions[0] = (self.positions[0] + 1) % 26
    
    def _encode_through_rotor(self, char_index: int, rotor_index: int, forward: bool = True) -> int:
        rotor = self.rotors[rotor_index]
        position = self.positions[rotor_index]
        ring = self.ring_settings[rotor_index]
        
        if forward:
            shifted = (char_index + position - ring) % 26
            encoded = self.ALPHABET.index(rotor[shifted])
            return (encoded - position + ring) % 26
        else:
            shifted = (char_index + position - ring) % 26
            encoded = rotor.index(self.ALPHABET[shifted])
            return (encoded - position + ring) % 26
    
    def _encode_through_reflector(self, char_index: int) -> int:
        return self.ALPHABET.index(self.reflector[char_index])
    
    def encrypt_char(self, char: str) -> str:
        if char not in self.ALPHABET:
            return char
        
        self._rotate_rotors()
        char = self._plugboard_swap(char)
        char_index = self.ALPHABET.index(char)
        
        for i in range(3):
            char_index = self._encode_through_rotor(char_index, i, forward=True)
        
        char_index = self._encode_through_reflector(char_index)
        
        for i in range(2, -1, -1):
            char_index = self._encode_through_rotor(char_index, i, forward=False)
        
        result = self.ALPHABET[char_index]
        result = self._plugboard_swap(result)
        return result
    
    def encrypt(self, text: str) -> str:
        return ''.join(self.encrypt_char(c.upper()) for c in text if c.isalpha())




#### Bombe Decoding Part
* New improvements
* has a sliding crib function which was the last ones main issue,
* basically it goes over each individual character up until it reaches a certain point
* did 500 but that sometimes takes way too long so can modify by either removing cribs
* or by increaseing the length of where it "slides" too

In [2]:
class Bombe:
    """
    Bombe with SLIDING CRIB SEARCH
    Can find cribs at ANY position in the text
    """
    
    MIN_CRIB_LENGTH = 8  # Minimum safe crib length = basically will igvev you a warning
    # if your crib is too small and could cause false positives
    
    def __init__(self, rotors_to_test: List[Tuple[str, str, str]] = None):
        if rotors_to_test is None:
            self.rotors_to_test = [
                ('I', 'II', 'III'),
                ('I', 'III', 'II'),
                ('II', 'I', 'III'),
                ('II', 'III', 'I'),
                ('III', 'I', 'II'),
                ('III', 'II', 'I')
            ]
        else:
            self.rotors_to_test = rotors_to_test
    
    def _test_crib(self, ciphertext: str, crib: str, 
                   rotors: Tuple[str, str, str],
                   rotor_positions: Tuple[int, int, int],
                   plugboard: Dict[str, str] = None) -> bool:
        try:
            enigma = EnigmaMachine(rotors, rotor_positions, plugboard=plugboard)
            encrypted_crib = enigma.encrypt(crib)
            return encrypted_crib == ciphertext[:len(crib)]
        except:
            return False
    
    def break_enigma(self, ciphertext: str, crib: str, 
                    crib_position: int = 0,
                    max_positions: int = 17576,
                    plugboard: Dict[str, str] = None,
                    verbose: bool = True) -> Optional[Dict]:
        """Attempt to break Enigma encryption using a known crib"""
        
        crib = crib.upper().replace(' ', '')
        ciphertext = ciphertext.upper().replace(' ', '')
        
        # Warn about short cribs
        if len(crib) < self.MIN_CRIB_LENGTH and verbose:
            print(f"      WARNING: Crib '{crib}' is only {len(crib)} letters")
            print(f"      Recommended minimum: {self.MIN_CRIB_LENGTH} letters")
            print(f"      Risk of false positives!")
        
        if len(crib) > len(ciphertext):
            raise ValueError("Crib longer than ciphertext")
        
        # Extract the ciphertext portion we're trying to match
        crib_ciphertext = ciphertext[crib_position:crib_position + len(crib)]
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"BOMBE CRYPTANALYSIS STARTING")
            print(f"{'='*70}")
            print(f"Crib: '{crib}' (length: {len(crib)})")
            print(f"Crib position: {crib_position}")
            print(f"Target ciphertext section: '{crib_ciphertext}'")
            print(f"Testing {len(self.rotors_to_test)} rotor combinations")
            print(f"Max positions to test per rotor config: {max_positions:,}")
            print(f"{'='*70}\n")
        
        start_time = time.time()
        tests_performed = 0
        
        for rotor_config in self.rotors_to_test:
            if verbose:
                print(f"Testing rotors: {rotor_config}")
            
            positions_tested = 0
            for pos in product(range(26), range(26), range(26)):
                if positions_tested >= max_positions:
                    if verbose:
                        print(f"  Reached max positions ({max_positions:,})")
                    break
                
                tests_performed += 1
                positions_tested += 1
                
                if self._test_crib(crib_ciphertext, crib, rotor_config, pos, plugboard):
                    elapsed = time.time() - start_time
                    
                    if verbose:
                        print(f"\n{'='*70}")
                        print(f"✓ SOLUTION FOUND!")
                        print(f"{'='*70}")
                        print(f"Rotors: {rotor_config}")
                        print(f"Positions: {pos} ({chr(65+pos[0])}{chr(65+pos[1])}{chr(65+pos[2])})")
                        print(f"Tests performed: {tests_performed:,}")
                        print(f"Time elapsed: {elapsed:.2f} seconds")
                        print(f"{'='*70}\n")
                    
                    return {
                        'rotors': rotor_config,
                        'positions': pos,
                        'positions_letters': f"{chr(65+pos[0])}{chr(65+pos[1])}{chr(65+pos[2])}",
                        'tests_performed': tests_performed,
                        'time_seconds': elapsed,
                        'crib': crib,
                        'crib_position': crib_position
                    }
                
                if verbose and tests_performed % 1000 == 0:
                    print(f"  Tested {tests_performed:,} settings...", end='\r')
        
        if verbose:
            elapsed = time.time() - start_time
            print(f"\n✗ No solution found after {tests_performed:,} tests in {elapsed:.2f} seconds")
        
        return None
    
    def break_with_sliding_cribs(self, ciphertext: str, 
                                 cribs: List[str],
                                 max_positions: int = 17576,
                                 max_slide_distance: int = 200,
                                 plugboard: Dict[str, str] = None,
                                 verbose: bool = True) -> Optional[Dict]:
        """
        NEW METHOD: Search for cribs at ANY position in text
        
        Args:
            ciphertext: The encrypted message
            cribs: List of crib strings (no positions needed!)
            max_positions: Maximum rotor positions to test per attempt
            max_slide_distance: How far into the text to search (limits for speed)
            plugboard: Known plugboard settings
            verbose: Print progress
        
        Returns:
            Dictionary with found settings or None
        """
        print(f"\n{'='*70}")
        print(f"SLIDING CRIB SEARCH")
        print(f"{'='*70}")
        print(f"Cribs to try: {len(cribs)}")
        print(f"Search mode: Crib can appear ANYWHERE in first {max_slide_distance} characters")
        print(f"{'='*70}\n")
        
        for crib_num, crib in enumerate(cribs, 1):
            crib = crib.upper().replace(' ', '')
            
            print(f"\n{'='*70}")
            print(f"CRIB {crib_num}/{len(cribs)}: '{crib}' (length: {len(crib)})")
            print(f"{'='*70}")
            
            # Check crib length
            if len(crib) < self.MIN_CRIB_LENGTH:
                print(f"    WARNING: Crib too short ({len(crib)} < {self.MIN_CRIB_LENGTH})")
                print(f"    Skipping to avoid false positives!")
                continue
            
            # Calculate how far we can slide the crib
            max_search_pos = min(len(ciphertext) - len(crib), max_slide_distance)
            
            print(f"Searching positions 0 to {max_search_pos}...")
            
            # Try crib at each position
            for pos in range(max_search_pos + 1):
                if verbose and pos % 20 == 0 and pos > 0:
                    print(f"  Checked {pos}/{max_search_pos} positions...", end='\r')
                
                result = self.break_enigma(
                    ciphertext=ciphertext,
                    crib=crib,
                    crib_position=pos,
                    max_positions=max_positions,
                    plugboard=plugboard,
                    verbose=False  # Silent for each position
                )
                
                if result:
                    print(f"\n\n{'='*70}")
                    print(f"✓✓✓ CRIB FOUND AT POSITION {pos}! ✓✓✓")
                    print(f"{'='*70}")
                    print(f"Crib: '{crib}'")
                    print(f"Position in text: {pos}")
                    print(f"Rotors: {result['rotors']}")
                    print(f"Positions: {result['positions_letters']}")
                    print(f"{'='*70}\n")
                    
                    result['successful_crib_number'] = crib_num
                    result['total_cribs_tested'] = crib_num
                    result['all_cribs'] = cribs
                    result['crib_found_at_position'] = pos
                    return result
            
            print(f"\n✗ Crib '{crib}' not found at any position (0-{max_search_pos})")
        
        return {
            'success': False,
            'total_cribs_tested': len(cribs),
            'all_cribs': cribs,
            'message': 'No solution found with any cribs at any position'
        }

#### Main thing to run after setting cribs and searching length

In [6]:
# Helper functions (same as before)
def decrypt_message(ciphertext: str, rotors: Tuple[str, str, str], 
                   positions: Tuple[int, int, int],
                   plugboard: Dict[str, str] = None) -> str:
    enigma = EnigmaMachine(rotors, positions, plugboard=plugboard)
    return enigma.encrypt(ciphertext)


def calculate_accuracy(decrypted_text: str, expected_text: str) -> float:
    decrypted = decrypted_text.upper().replace(' ', '')
    expected = expected_text.upper().replace(' ', '')
    
    min_len = min(len(decrypted), len(expected))
    if min_len == 0:
        return 0.0
    
    matches = sum(1 for i in range(min_len) if decrypted[i] == expected[i])
    return (matches / min_len) * 100


def save_results(result, decrypted, ciphertext, original, accuracy, actual_settings, output_file):
    """Save results to file"""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write("BOMBE CRYPTANALYSIS RESULTS (SLIDING CRIB SEARCH)\n")
        f.write("="*80 + "\n\n")
        f.write(f"Timestamp: {timestamp}\n\n")
        
        f.write("-"*80 + "\n")
        f.write("SETTINGS FOUND BY BOMBE\n")
        f.write("-"*80 + "\n")
        f.write(f"Rotor Configuration: {result['rotors']}\n")
        f.write(f"Rotor Positions (numeric): {result['positions']}\n")
        f.write(f"Rotor Positions (letters): {result['positions_letters']}\n\n")
        
        if 'crib_found_at_position' in result:
            f.write(f"Crib '{result['crib']}' found at position: {result['crib_found_at_position']}\n\n")
        
        if actual_settings:
            f.write("-"*80 + "\n")
            f.write("ACTUAL ENCRYPTION SETTINGS\n")
            f.write("-"*80 + "\n")
            f.write(f"Rotor Configuration: {actual_settings['rotors']}\n")
            f.write(f"Rotor Positions (numeric): {actual_settings['positions']}\n")
            f.write(f"Rotor Positions (letters): {actual_settings['positions_letters']}\n\n")
            
            rotors_match = result['rotors'] == actual_settings['rotors']
            positions_match = result['positions'] == actual_settings['positions']
            
            f.write("VERIFICATION: ")
            f.write(f"{'✓ CORRECT' if (rotors_match and positions_match) else '✗ INCORRECT'}\n\n")
        
        f.write("-"*80 + "\n")
        f.write("PERFORMANCE\n")
        f.write("-"*80 + "\n")
        f.write(f"Time: {result['time_seconds']:.2f} seconds\n")
        f.write(f"Tests: {result['tests_performed']:,}\n")
        f.write(f"Accuracy: {accuracy:.2f}%\n\n")
        
        f.write("-"*80 + "\n")
        f.write("DECRYPTED TEXT (first 500 chars)\n")
        f.write("-"*80 + "\n")
        f.write(f"{decrypted[:500]}\n\n")
    
    print(f"✓ Results saved to: {output_file}")


def main():
    print("\n" + "="*80)
    print("BOMBE WITH SLIDING CRIB SEARCH")
    print("="*80 + "\n")
    
#####################################################################################################
    
    encrypted_file = r"C:\Users\tapia\Desktop\Python stuff\Classes\Capstone\training data\Encrypted Processed Discussion1.txt"      # Your encrypted file
    plaintext_file = r"C:\Users\tapia\Desktop\Python stuff\Classes\Capstone\training data\Processed Discussion1.txt"      # Original plaintext
    output_file = "bombe_results.txt"

#####################################################################################################
    actual_settings = {
        'rotors': ('I', 'II', 'III'),
        'positions': (2, 25, 19),
        'positions_letters': 'CZT'
    }
    
    # Read files
    if not os.path.exists(encrypted_file):
        print(f"ERROR: {encrypted_file} not found!")
        return
    
    with open(encrypted_file, 'r', encoding='utf-8') as f:
        ciphertext = f.read().strip()
    ciphertext = ''.join(c for c in ciphertext if c.isalpha()).upper()
    
    original_plaintext = ""
    if os.path.exists(plaintext_file):
        with open(plaintext_file, 'r', encoding='utf-8') as f:
            original_plaintext = f.read().strip()
        original_plaintext = ''.join(c for c in original_plaintext if c.isalpha()).upper()
    
    print(f"Ciphertext length: {len(ciphertext)} characters\n")
    
    # CRIBS - 
    cribs = [
        "ICHOSETHECONCERNS",        
    ]
    
    print("Cribs to search for:")
    for i, crib in enumerate(cribs, 1):
        print(f"  {i}. '{crib}' ({len(crib)} letters)")
    print()
    
###################################################################################################################################################
    bombe = Bombe()
    result = bombe.break_with_sliding_cribs(
        ciphertext=ciphertext,
        cribs=cribs,
        max_positions=17576, #basically this is all of the rotor positions = 26^3
        max_slide_distance=40,  # Search first until whatever point you choose where each "point" is up to a certain character in the text
        verbose=True
    )
###################################################################################################################################################
    
    if result and result.get('success', True):
        print("\n" + "="*80)
        print("✓ SUCCESS!")
        print("="*80)
        
        decrypted = decrypt_message(ciphertext, result['rotors'], result['positions'])
        accuracy = calculate_accuracy(decrypted, original_plaintext) if original_plaintext else 0.0
        
        save_results(result, decrypted, ciphertext, original_plaintext, 
                    accuracy, actual_settings, output_file)
        
        print(f"\nSettings: {result['rotors']} @ {result['positions_letters']}")
        print(f"Accuracy: {accuracy:.2f}%")
        print(f"Time: {result['time_seconds']:.2f}s")
        print("="*80 + "\n")
    else:
        print("\n✗ Failed to find solution")


if __name__ == "__main__":
    main()



BOMBE WITH SLIDING CRIB SEARCH

Ciphertext length: 2759 characters

Cribs to search for:
  1. 'ICHOSETHECONCERNS' (17 letters)


SLIDING CRIB SEARCH
Cribs to try: 1
Search mode: Crib can appear ANYWHERE in first 40 characters


CRIB 1/1: 'ICHOSETHECONCERNS' (length: 17)
Searching positions 0 to 40...
  Checked 20/40 positions...

✓✓✓ CRIB FOUND AT POSITION 21! ✓✓✓
Crib: 'ICHOSETHECONCERNS'
Position in text: 21
Rotors: ('I', 'II', 'III')
Positions: XAT


✓ SUCCESS!
✓ Results saved to: bombe_results.txt

Settings: ('I', 'II', 'III') @ XAT
Accuracy: 4.20%
Time: 0.73s

